# Module 1: Agentic RAG


## 1. Intro

$\theta^* = \arg\max_{\theta} \sum_{t} \log P(x_{t+1} \mid x_{\le t};\, \theta)$

Find the model parameters $\theta$ such that the training text is as predictable as possible. At every position $t$, the model sees the context thus far $x_{\le t}$, and assigns a probability to the token that actually comes next $x_{t+1}$. The best parameters $\theta^*$ are the ones that maximize the sum of those log-probabilities across the training data.

In other words, a large language model (LLM) is a system that learns patterns from vast amounts of text to generate context-aware language.

## 2. Environment

In [ ]:
import json
import requests
import time

from dotenv import load_dotenv
from minsearch import Index
from openai import OpenAI
from pathlib import Path
from sqlitesearch import TextSearchIndex
from textwrap import dedent
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

from ingest import build_index, load_faq_data
from rag_helper import RAGBase

In [ ]:
load_dotenv()
openai_client = OpenAI()

## 3. RAG

Retrieval-Augmented Generation (RAG) is a technique that extends an LLM by giving it access to external information at inference time.

Instead of relying solely on what was learned during training, the model first **retrieves** relevant documents from an external source, then **generates** a response conditioned on both the query and the retrieved content.

In other words, RAG lets a language model retrieve relevant information before generating an answer, making its responses more grounded, specific, and up to date.

## 4. Dataset

FAQ data is available as JSON from the [DataTalks.Club](https://datatalks.club/faq/) website. In the RAG pipeline, this dataset is used as the knowledge base:
1. All the documents are indexed
2. When a user makes a query, we search the index
3. The search returns the most relevant FAQ entries
4. Those entries are passed to the LLM as context
5. The LLM generates an answer based on the context

In practice, data preparation is often the most time-consuming part of building a RAG system.

In [ ]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

## 5. Search

A search engine is a system that finds the most relevant documents for a query.

Formally: $s_i = \operatorname{sim}(q, d_i)$

For each document $d_i$, the search engine computes a similarity score $s_i$ against the query $q$, ranks all documents by that score, and returns the top results. The similarity function varies by approach: text search matches words, vector search matches meaning.

In other words, a search engine compares your query against many documents and returns the ones that match best.

### Indexing with minisearch


In [ ]:
index = Index(text_fields=["question", "section", "answer"], keyword_fields=["course"])
index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5,
)
search_results

In [ ]:
[doc["question"] for doc in search_results]

In [ ]:
# Boosting fields (i.e., importance)
results = index.search(
    question, num_results=5, boost_dict={"question": 2.0, "section": 0.5}
)

# Filtering by course (i.e., relevance)
results = index.search(
    question, num_results=5, filter_dict={"course": "mlops-zoomcamp"}
)

[doc["question"] for doc in results]

In [ ]:
def search(query: str) -> dict[str, str]:
    """Search the FAQ database for entries matching the given query."""
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )

## 6. Building the Prompt

When building AI systems, a prompt is usually split into two parts:
1. Instructions (system prompt): tells the LLM how to behave; does not change and stays the same for every request.
2. User prompt: includes the actual question and the retrieved context; changes with every request.

Note: Instructions ground the answer in the data and help to reduce hallucinations.

In [ ]:
INSTRUCTIONS = dedent("""
    Your task is to answer questions from the course participants
    based on the provided context.
    
    Use the context to find relevant information and provide accurate
    answers. If the answer is not found in the context,
    respond with "I don't know."
""").strip()

USER_PROMPT_TEMPLATE = dedent("""
    Question: {question}
    
    Context: {context}
""").strip()

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

## 7. LLM


In [ ]:
response = openai_client.responses.create(model="gpt-5.4-mini", input=prompt)
response.output_text

In [ ]:
response.usage

In [ ]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price
    + response.usage.output_tokens * output_price
)
cost

In [ ]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt},
]

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(model=model, input=message_history)

    return response.output_text


def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    
    return answer

In [ ]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

In [ ]:
print(rag("How do I get a certificate?"))

## 8. RAG Helper


In [ ]:
documents = load_faq_data()
index = build_index(documents)

custom_instructions = dedent("""
    You're a course teaching assistant.
    Answer the QUESTION based on the CONTEXT from the FAQ database.
    Use only the facts from the CONTEXT when answering the QUESTION.
""").strip()

assistant = RAGBase(
    index=index, llm_client=openai_client, instructions=custom_instructions
)
index

In [ ]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

In [ ]:
print(assistant.rag("How do I get a certificate?"))

## 9. Data Ingestion

Persistent search lets the index survive beyond a single application run. Instead of rebuilding the index at startup, we separate ingestion from querying: one process prepares and writes the data to a durable index, and another process reads from it during search.

This is preferred when datasets are large, data preparation is slow, or the service needs fast restarts. Common backends include Elasticsearch, OpenSearch, Qdrant, and SQLite FTS5.

In [ ]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"Loaded {len(documents)} documents\nLLM Zoomcamp: {len(docs_llm)} documents")

In [ ]:
db_path = Path("faq.db")

if db_path.exists():
    print("Index already exists. Skipping.")
else:
    index = TextSearchIndex(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"],
        db_path=str(db_path),
    )

    for doc in docs_llm:
        index.add(doc)
        print(f'Added: {doc["question"][:60]}...')
        time.sleep(0.5)
        
    index.close()
    print("Done. Index saved to faq.db")

In [ ]:
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)
sqlite_index.count()

In [ ]:
results = sqlite_index.search(
    "Can I still join the course after it started?", num_results=5
)

[doc["question"] for doc in results]

In [ ]:
assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

In [ ]:
sqlite_index.close()

## 10. Next Steps

After building a basic RAG pipeline, the next steps are improving retrieval and orchestration. Agents let the LLM decide how to search, vector search improves semantic matching, and production backends like Elasticsearch or OpenSearch make search more scalable. RAG is usually preferred over fine-tuning because it is easier to update, cheaper to maintain, and works with any LLM.

## 11. Agents

Agents make RAG pipelines more flexible by letting the LLM decide what to do next. Instead of always running one fixed search, the LLM can reformulate the query, search again, ask for clarification, use tools, or stop and answer. This is useful when basic lexical search fails because of typos, unusual wording, or questions that need multiple retrieval steps.

## 12. Quick RAG Revision


In [ ]:
print(assistant.rag("How do I run Ollama locally?"))

In [ ]:
print(assistant.rag("How do I run Olama locally?"))

## 13. Function Calling

Function calling lets an LLM request external tools in a structured way. Instead of only producing text, the model can ask the program to run a specific function, such as searching an index, calling an API, or running a calculation.

In [ ]:
messages = [{"role": "user", "content": "I just discovered the course. Can I join it?"}]

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)
response.output

In [ ]:
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

messages.extend(response.output)
messages.append(
    {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }
)

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)
response.output_text

In [ ]:
usage = response.usage
usage.input_tokens, usage.output_tokens

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

## 14. The Agentic Loop

An agentic loop turns function calling into a repeatable process. The LLM chooses a tool call, the program runs it, the result is added to memory, and the LLM decides whether to call another tool or give the final answer.

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
instructions = dedent("""
    You're a course teaching assistant.
    You're given a question from a course student and your task is to answer it.
    
    If you want to look up information, use the search function. 
    Use as many keywords from the user question as possible when making first requests.
    
    Make multiple searches.
    
    Try to expand your search by using new keywords
    based on the results you get from the search.
    
    At the end, ask if there are other areas that the user wants to explore.
""").strip()

question = "I just discovered the course. Can I join it?"

In [ ]:
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)

has_function_calls = False
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model, input=messages, tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it += 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

In [ ]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

In [ ]:
instructions = dedent("""
    You're a course teaching assistant.
    You're given a question from a course student and your task is to answer it.
    
    If you want to look up information, use the search function. 
    Use as many keywords from the user question as possible when making first requests.
    
    Make multiple searches. First perform search, analyze the results 
    and then perform more searches. 
    
    At the end, ask if there are other areas that the user wants to explore.
""").strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

In [ ]:
instructions = dedent("""
    You're a course teaching assistant.
    You're given a question from a course student and your task is to answer it.
    
    If you want to look up information, use the search function. 
    Use as many keywords from the user question as possible when making first requests.
    
    Make multiple searches. First perform search, analyze the results 
    and then perform more searches. 
    
    The question has to be about the course or its logistics, offtopic questions 
    shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
    If you can't answer the question using FAQ, don't do it yourself. Only use the 
    facts from the FAQ database.
    
    At the end, ask if there are other areas that the user wants to explore.
""").strip()

agent_loop(instructions, "What is Queen's gambit?")

## 15. Toy AI Kit

Agent frameworks wrap the repetitive parts of the agentic loop. You define the tools, prompts, and LLM client; the framework handles schema generation, tool execution, message history, repeated model calls, and cost tracking. ToyAIKit is used here as a simple teaching framework, not a production framework.

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

In [ ]:
result.cost

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

In [ ]:
runner.run()  # interactive chat

## 16. Other Frameworks

* OpenAI Agents SDK: official OpenAI agent framework, good if you’re already using the Responses API.
* PydanticAI: type-safe, multi-provider, plain Python functions as tools; the author’s preferred option.
* LangChain / LangGraph: popular, many integrations, useful for complex workflows.
* Google ADK: good for Gemini / Google Cloud stacks.
* Others mentioned: CrewAI, AutoGen, Semantic Kernel, Smolagents, and Anthropic Tool Use.

Main advice:
* Don’t use agents unless needed.
* Start with simpler patterns first: RAG, one LLM call, parsing, or code.
* Use agents only for multi-step tool use / decisions.
* Agents add cost, latency, complexity, and unpredictability.
* Frameworks mainly differ in ergonomics and integrations.